# 07 — Data Model Summary

Consolidated reference of all ESPN Fantasy Basketball API data structures.
This notebook serves as the bridge to the full-stack Nikola build.

In [ ]:
import sys
sys.path.insert(0, "../src")
from connection import get_league
import pandas as pd

league = get_league()
print(f"Connected: {league.settings.name} ({league.year})")
print(f"Scoring type: {league.settings.scoring_type}")

## 1. League Object

| Property | Type | Description |
|----------|------|-------------|
| `teams` | `List[Team]` | All teams in the league |
| `settings` | `Settings` | League config (scoring type, roster slots, etc.) |
| `currentMatchupPeriod` | `int` | Current matchup week |
| `scoringPeriodId` | `int` | Current scoring period (day) |
| `firstScoringPeriod` | `int` | Season start |
| `finalScoringPeriod` | `int` | Season end |
| `current_week` | `int` | Current week |
| `pro_schedule` | `dict` | NBA team schedules by team ID |
| `matchup_ids` | `dict` | Matchup period → scoring period mapping |
| `player_map` | `dict` | Player ID ↔ name mapping |

### Key Methods
| Method | Returns | Notes |
|--------|---------|-------|
| `standings()` | `List[Team]` | Sorted by standing |
| `scoreboard(matchupPeriod)` | `List[Matchup]` | Current or specified week |
| `box_scores(matchup_period)` | `List[BoxScore]` | Detailed per-player stats |
| `free_agents(position, size)` | `List[Player]` | Filter by position |
| `recent_activity(size, msg_type)` | `List[Activity]` | FA/WAIVER/TRADED |
| `transactions(types)` | `List[Transaction]` | Richer transaction data |
| `player_info(name, playerId)` | `Player` | Lookup individual player |

In [ ]:
# Verify league-level data
print(f"Teams:              {len(league.teams)}")
print(f"Matchup period:     {league.currentMatchupPeriod}")
print(f"Scoring period:     {league.scoringPeriodId}")
print(f"Season:             periods {league.firstScoringPeriod} - {league.finalScoringPeriod}")
print(f"Pro schedule teams: {len(league.pro_schedule)}")
print(f"Player map size:    {len(league.player_map)}")

## 2. Team Object

| Property | Type | Description |
|----------|------|-------------|
| `team_id` | `int` | Unique team ID |
| `team_name` | `str` | Full team name |
| `team_abbrev` | `str` | Abbreviation |
| `wins/losses/ties` | `int` | Record |
| `standing` | `int` | Current standing/seed |
| `roster` | `List[Player]` | Current roster |
| `schedule` | `List[Matchup]` | Season matchups |
| `acquisitions` | `int` | Total add/drops |
| `trades` | `int` | Total trades |

In [ ]:
t = league.teams[0]
print(f"Sample team: {t.team_name}")
print(f"  Record: {t.wins}-{t.losses}-{t.ties}")
print(f"  Roster size: {len(t.roster)}")
print(f"  Acquisitions: {t.acquisitions}")

## 3. Player Object

| Property | Type | Description |
|----------|------|-------------|
| `name` | `str` | Player name |
| `playerId` | `int` | ESPN player ID |
| `position` | `str` | Primary position |
| `eligibleSlots` | `list` | All eligible lineup slots |
| `lineupSlot` | `str` | Current lineup slot |
| `proTeam` | `str` | NBA team abbreviation |
| `injuryStatus` | `str` | ACTIVE, OUT, QUESTIONABLE, etc. |
| `stats` | `dict` | Nested stats (see key scheme below) |
| `schedule` | `dict` | Games by scoring period |
| `total_points` | `float` | Season total fantasy points |
| `avg_points` | `float` | Per-game average |
| `nine_cat_averages` | `dict` | 9-cat averages |

### Stats Key Scheme

ESPN raw API uses numeric keys like `002025`. The library translates:

| Library Key | ESPN Raw | Description |
|------------|----------|-------------|
| `{year}_total` | `00{year}` | Season totals |
| `{year}_projected` | `10{year}` | Season projections |
| `{period}_last_7` | `01{year}` | Last 7 days |
| `{period}_last_15` | `02{year}` | Last 15 days |
| `{period}_last_30` | `03{year}` | Last 30 days |

Each entry contains:
```python
{
    'applied_total': float,
    'applied_avg': float,
    'avg': {'PTS': x, 'REB': x, 'AST': x, ...},
    'total': {'PTS': x, 'REB': x, 'AST': x, ...}
}
```

### Nine-Cat Stats
`PTS`, `REB`, `AST`, `STL`, `BLK`, `3PM`, `FG%`, `FT%`, `TO`

**Important: Turnovers (TO) are inverted — lower is better.**

In [ ]:
p = t.roster[0]
print(f"Sample player: {p.name}")
print(f"  Stats keys: {list(p.stats.keys())}")
print(f"  Nine-cat averages: {p.nine_cat_averages}")

## 4. BoxScore (H2H Category)

| Property | Type | Description |
|----------|------|-------------|
| `home_team` / `away_team` | `Team` | Teams in matchup |
| `home_stats` / `away_stats` | `dict` | `{cat: {value, result}}` |
| `home_wins/losses/ties` | `int` | Category record |
| `home_lineup` / `away_lineup` | `List[BoxPlayer]` | Per-player data |

### BoxPlayer (extends Player)
| Property | Type | Description |
|----------|------|-------------|
| `slot_position` | `str` | Lineup slot (PG, SG, UT, BE, IR) |
| `pro_opponent` | `str` | Opposing NBA team |
| `game_played` | `int` | % of game played (0-100) |
| `points` | `float` | Fantasy points scored |
| `points_breakdown` | `dict` | Stat-by-stat breakdown |

In [ ]:
bs = league.box_scores(matchup_period=1)[0]
print(f"BoxScore type: {type(bs).__name__}")

if hasattr(bs, 'home_stats') and bs.home_stats:
    print(f"\nCategories tracked: {list(bs.home_stats.keys())}")
    print(f"Sample stat entry: {list(bs.home_stats.items())[0]}")
    print(f"\nHome lineup size: {len(bs.home_lineup)}")
    bp = bs.home_lineup[0]
    print(f"BoxPlayer sample: {bp.name}, slot={bp.slot_position}, opponent={bp.pro_opponent}")

## 5. Matchup Object (from scoreboard)

| Property | Type | Description |
|----------|------|-------------|
| `home_team` / `away_team` | `Team` | Matchup teams |
| `home_final_score` / `away_final_score` | `float` | Final scores |
| `home_team_cats` / `away_team_cats` | `dict` | `{cat: {score, result}}` |
| `winner` | `str` | HOME, AWAY, or TIE |

In [ ]:
m = league.scoreboard()[0]
print(f"Matchup: {m.home_team.team_name} vs {m.away_team.team_name}")
print(f"  Score: {m.home_final_score} - {m.away_final_score}")
print(f"  Winner: {m.winner}")
if m.home_team_cats:
    print(f"  Category results: {m.home_team_cats}")

## 6. Activity & Transactions

### Activity (`recent_activity`)
| Property | Type | Description |
|----------|------|-------------|
| `actions` | `list` | `[(team, action, player, position), ...]` |
| `date` | `int` | Timestamp (ms) |

Action types: `FA ADDED`, `WAIVER ADDED`, `DROPPED`, `TRADED`, `MOVED`

### Transaction (`transactions`)
| Property | Type | Description |
|----------|------|-------------|
| `team` | `Team` | Team making the transaction |
| `type` | `str` | FREEAGENT, WAIVER, TRADE_ACCEPT, etc. |
| `status` | `str` | Transaction status |
| `scoring_period` | `int` | When it occurred |
| `date` | `int` | Process date (timestamp ms) |
| `bid_amount` | `int` | Waiver bid (None for FA) |
| `items` | `List[TransactionItem]` | ADD/DROP items |

In [ ]:
acts = league.recent_activity(size=5)
print(f"Recent activity: {len(acts)} entries")
for act in acts[:3]:
    for team, action, player, pos in act.actions:
        tn = team.team_name if hasattr(team, 'team_name') else str(team)
        print(f"  {tn:25s} {action:15s} {player}")

## 7. Roster Slot Configuration

Lineup positions available in this league (from POSITION_MAP):

| ID | Slot | Description |
|----|------|-------------|
| 0 | PG | Point Guard |
| 1 | SG | Shooting Guard |
| 2 | SF | Small Forward |
| 3 | PF | Power Forward |
| 4 | C | Center |
| 5 | G | Guard |
| 6 | F | Forward |
| 11 | UT | Utility |
| 12 | BE | Bench |
| 13 | IR | Injured Reserve |

In [ ]:
# Count actual slots in use across all teams
from collections import Counter

slot_counts = Counter()
for t in league.teams:
    for p in t.roster:
        slot_counts[p.lineupSlot] += 1

print("Roster slot distribution (across all teams):")
for slot, count in slot_counts.most_common():
    per_team = count / len(league.teams)
    print(f"  {slot:5s}: {count:3d} total ({per_team:.0f} per team)")

## 8. Key Observations for Full-Stack Build

### Data Refresh Strategy
- `league.fetch_league()` refreshes all data
- `scoringPeriodId` increments daily
- `currentMatchupPeriod` increments weekly
- `matchup_ids` maps weekly matchup periods to daily scoring periods

### Weekly Lineup Optimization
- `pro_schedule` has NBA games by scoring period
- Player's `schedule` attribute for game availability
- `eligibleSlots` determines which lineup slots a player can fill
- Game limit: 25 games/week (per plan.md: optimize for 24+7 strategy)
- Acquisition limit: 7 per week

### Monitoring & Alerts (for Telegram bot)
- `recent_activity()` for competitor add/drops
- `injuryStatus` for player health changes
- `free_agents()` for streaming targets
- Box score category results for matchup tracking

### H2H Category Strategy
- 9 categories: PTS, REB, AST, STL, BLK, 3PM, FG%, FT%, TO
- Win 5 of 9 categories to win the matchup
- TO is inverted (lower = better)
- FG% and FT% are volume-weighted (not simple averages)

In [ ]:
print("Phase 1 exploration complete.")
print(f"\nLeague: {league.settings.name}")
print(f"Season: {league.year}")
print(f"Teams: {len(league.teams)}")
print(f"Current week: {league.current_week}")
print(f"Scoring type: {league.settings.scoring_type}")